# Underwater Object Detection

Training a Faster Region-based Covolutional Neural Network model for underwater object detection. \
Dataset: [Underwater Image Dataset](https://www.kaggle.com/datasets/slavkoprytula/aquarium-data-cots). 

## Introduction 
Object detection is a computer vision task that involves both localizing objects in images and classifying them. Faster R-CNN is a popular deep learning model for object detection that is an evolution of the R-CNN and Fast R-CNN models. Faster R-CNN is composed of two modules: a region proposal network (RPN) that generates region proposals and a network that uses these proposals to detect objects. You can learn more about this network [here](https://arxiv.org/abs/1506.01497). The architecture used in this notebook is based on [this paper](https://www.researchgate.net/publication/368708716_Underwater_Object_Detection_Method_Based_on_Improved_Faster_RCNN).

In this notebook we will use some techniques to improve the performance of the model, such as data augmentation, transfer learning, and hyperparameter tuning, OHEM, GIOU loss, and Soft-NMS.

- **Data Augmentation**: Data augmentation is a technique used to artificially increase the size of the training dataset by applying transformations to the images. This technique can help the model generalize better to new data and improve its performance.
- **Transfer Learning**: Transfer learning is a technique that allows you to use a pre-trained model as a starting point for training a new model on a different task. This can help you achieve better performance with less data and computational resources.
- **Hyperparameter Tuning**: Hyperparameter tuning is the process of finding the best set of hyperparameters for a machine learning model. This can help you improve the performance of the model and reduce the risk of overfitting.
- **OHEM**: Online Hard Example Mining (OHEM) is a technique used to focus the training process on the most challenging examples in the dataset. This can help the model learn from its mistakes and improve its performance.
- **GIOU Loss**: Generalized Intersection over Union (GIOU) is a loss function used to measure the similarity between two bounding boxes. This loss function can help the model learn to predict more accurate bounding boxes.
- **Soft-NMS**: Soft Non-Maximum Suppression (Soft-NMS) is a technique used to suppress overlapping bounding boxes by reducing the confidence scores of nearby boxes. This can help the model produce more accurate detections.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from collections import Counter
import seaborn as sns
import tqdm
import glob
import os
import json
from PIL import Image
import xml.etree.ElementTree as ET
import pprint
pp = pprint.PrettyPrinter(indent=4)
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
import torchvision.ops as ops
from torch.utils.data import distributed, RandomSampler, SequentialSampler
import random
import cv2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## EDA (Exploratory Data Analysis)

We will start by loading the dataset and exploring its contents. \
The dataset contains images of underwater scenes with annotations for the objects present in the images. \
We will load the images and annotations, visualize some examples, and analyze the distribution of object classes in the dataset.

In [ ]:
TRAIN_PATH = "aquarium_pretrain/train"
TEST_PATH = "aquarium_pretrain/test"
VAL_PATH = "aquarium_pretrain/val"

TRAIN_IMAGE_PATH = "aquarium_pretrain/train_images"
TEST_IMAGE_PATH = "aquarium_pretrain/test_images"
VAL_IMAGE_PATH = "aquarium_pretrain/val_images"

TRAIN_LABEL_PATH = "aquarium_pretrain/train_labels"
TEST_LABEL_PATH = "aquarium_pretrain/test_labels"
VAL_LABEL_PATH = "aquarium_pretrain/val_labels"

data = "aquarium_pretrain/data.yaml"